#📌 Extracción

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_ml = pd.get_dummies(df_telecom_listo, columns=variables_categoricas, drop_first=True)

print("Proporción original de la clase Evasion:")
print(df_ml['Evasion'].value_counts(normalize=True) * 100)

X = df_ml.drop('Evasion', axis=1)
y = df_ml['Evasion']

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_balanceado, y_balanceado = smote.fit_resample(X, y)

print("\nProporción después de aplicar SMOTE:")
print(y_balanceado.value_counts(normalize=True) * 100)

#🔧 Transformación

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X_balanceado, y_balanceado, test_size=0.20, random_state=42)

scaler = StandardScaler()
columnas_numericas = ['Meses_Contrato', 'Cargo_Mensual', 'Cargo_Total', 'Cuentas_Diarias']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[columnas_numericas] = scaler.fit_transform(X_train[columnas_numericas])
X_test_scaled[columnas_numericas] = scaler.transform(X_test[columnas_numericas])

#📊 Carga y análisis

In [ ]:
plt.figure(figsize=(12, 8))

correlacion = df_ml.corr()
sns.heatmap(correlacion[['Evasion']].sort_values(by='Evasion', ascending=False),
            annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlación de las variables con la Evasión')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Evasion', y='Meses_Contrato', data=df_telecom_listo, palette='Set2', ax=axes[0])
axes[0].set_title('Tiempo de Contrato vs Cancelación')

sns.boxplot(x='Evasion', y='Cargo_Total', data=df_telecom_listo, palette='Set2', ax=axes[1])
axes[1].set_title('Gasto Total vs Cancelación')

plt.tight_layout()
plt.show()

#Evaluación de modelos

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

modelo_lr = LogisticRegression(max_iter=1000, random_state=42)
modelo_lr.fit(X_train_scaled, y_train)
predicciones_lr = modelo_lr.predict(X_test_scaled)

modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)
predicciones_rf = modelo_rf.predict(X_test)

def evaluar_modelo(nombre, y_real, y_pred):
    print(f"--- Evaluación del Modelo: {nombre} ---")
    print(f"Exactitud (Accuracy): {accuracy_score(y_real, y_pred):.4f}")
    print(f"Precisión (Precision): {precision_score(y_real, y_pred):.4f}")
    print(f"Sensibilidad (Recall): {recall_score(y_real, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_real, y_pred):.4f}\n")

    cm = confusion_matrix(y_real, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de Confusión - {nombre}')
    plt.ylabel('Valor Real')
    plt.xlabel('Predicción')
    plt.show()

evaluar_modelo("Regresión Logística", y_test, predicciones_lr)
evaluar_modelo("Random Forest", y_test, predicciones_rf)

#Análisis de importancia de variables

In [ ]:
importancias = modelo_rf.feature_importances_
nombres_variables = X_train.columns

df_importancia = pd.DataFrame({'Variable': nombres_variables, 'Importancia': importancias})
df_importancia = df_importancia.sort_values(by='Importancia', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importancia', y='Variable', data=df_importancia.head(10), palette='viridis')
plt.title('Top 10 Variables más importantes para predecir Cancelación (Random Forest)')
plt.show()

coeficientes = modelo_lr.coef_[0]
df_coef = pd.DataFrame({'Variable': nombres_variables, 'Coeficiente': coeficientes})
df_coef = df_coef.sort_values(by='Coeficiente', ascending=False)

print("\nFactores que MÁS aumentan el riesgo de cancelación (Regresión Logística):")
print(df_coef.head(3))
print("\nFactores que MÁS protegen contra la cancelación (Regresión Logística):")
print(df_coef.tail(3))

#📄Informe final

#Desemepeño y comparación de modelos
Al comparar Random Forest y Regresión Logística, verificamos las métricas en Train y Test. Si Random Forest tiene 99% en Train y 80% en Test, presenta un ligero overfitting por su alta complejidad. Regresión Logística mostró resultados más estables (generaliza mejor) al ser un modelo más simple.

#Principales factores que afectan la cancelación
Basado en el gráfico de importancia y los coeficientes de la Regresión Logística, los factores determinantes son:

- Meses de Contrato (Tenure): Los clientes más nuevos son los que tienen mayor riesgo de abandonar.

- Tipo de Contrato: El contrato mes a mes (Month-to-month) tiene una altísima correlación positiva con la cancelación.

- Cargos Mensuales/Totales: Los clientes con altas cuotas mensuales tienden a buscar alternativas si no perciben valor.

#Estrategias de retención propuestas
- Migración de Contratos: Diseñar campañas agresivas de upselling ofreciendo un primer mes gratuito o descuentos si el cliente cambia su contrato de "Mes a mes" a "Anual".

- Programa de Onboarding: Como los clientes nuevos (pocos meses) son los que más se van, crear un servicio al cliente VIP durante los primeros 90 días para asegurar su satisfacción y adaptación al servicio.